# NB-17 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-17**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-17.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This replays the controlled bid-book capstone: screening and sizing lead to bid blocks, uncertain awards, and cash flows. Requested quantity is not automatically an owned position.

- [Volume VI, Chapter 12 — From an eligible path universe to a submitted FTR bid book](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c12)
- [Volume IV, Chapter 21 — The FTR laboratory](https://sherandoresearch.github.io/power-markets-series/vol4_testbed_atlas.html#chiv-21)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-17', require_key=True)
print(setup_status)


# NB-17 — Replay and stress the controlled FTR bid book
Companion to Volume VI §12 and Volume IV §21.6–21.11.
Prerequisites: NB-15 geometry and NB-06 risk diagnostics. This tutorial follows
the existing 650-path archive through screening, sizing, bid blocks, awards,
and held-out cash flows. It uses no new observations or random draws.

1. Verify and replay the released book before changing risk definitions.
2. Inspect the continuous certificates and historical selection ablation.
3. Reconcile corrected empirical ES, requested/awarded MW, and gross/net cash.
4. Vary one policy parameter at a time on the same retained scenarios.

In [ ]:
NOTEBOOK_ID='NB-17'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

a={k:np.asarray(v) for k,v in inputs['raw_inspection'].items()}
reference=inputs['original_legacy_reference_for_verification']
METHOD_VERSION='empirical-es-v2'
returned={};composed_requests={}
def run_saved(name,request=None):
    entry=inputs['requests'][name]
    q=copy.deepcopy(entry['request']) if request is None else request
    returned[name]=execute(name,entry['method'],q,entry['parents'])
    composed_requests[name]=q
    return returned[name]
def selected_carrier(name):
    entry=inputs['requests'][name];q=copy.deepcopy(entry['request'])
    parent=returned[entry['parents'][0]]['candidate_selection']
    screen=copy.deepcopy(parent['screen']);selected=screen['candidate_indices']
    original=[parent['original_indices'][j] for j in selected]
    for field in ('mean_gross','std_gross','market_price','market_price_std','forecast_alpha','standalone_score','reference_wtp','reference_fill_probability','reference_mean_fill_fraction','effective_requested_alpha','topology_instability'):
        screen[field]=[screen[field][j] for j in selected]
    for field in ('auction_prices','residual_path_capacity','reference_fill_fraction'):
        screen[field]=[[row[j] for j in selected] for row in screen[field]]
    screen['candidate_indices']=list(range(71,-1,-1)) if q['parameters']['ordering']=='reversed_candidates' else list(range(72))
    selection=q['inputs']['candidate_selection']
    assert selection['screen']=={} and selection['original_path_indices']==selection['selected_path_ids']==[]
    selection.update(screen=screen,original_path_indices=original,selected_path_ids=parent['screen']['candidate_path_ids'])
    panel=q['inputs']['signal_panel']
    for key in ('path_ids','sources','sinks'):
        assert panel[key]==[];panel[key]=a[key][original].tolist()
    for key,raw_key in (('train','product_train'),('test','product_test')):
        assert panel[key]==[];panel[key]=a[raw_key][:,original].tolist()
    for exposure,raw_key in zip(q['inputs']['scenario_exposures'],('exposure_intact','exposure_outage')):
        assert exposure['path_ids']==exposure['values']==[]
        exposure['path_ids']=panel['path_ids'];exposure['values']=a[raw_key][:,original].tolist()
    return q
strategies=('independent_top_k','column_addition_mean_variance','topology_robust_cvar')
for order in ('baseline','reversed_paths'):
    run_saved('screen:'+order)
    for mode in ('reference','construct'):
        variants=[order]+(['reversed_candidates'] if mode=='reference' and order=='baseline' else [])
        for variant in variants:
            for strategy in strategies:
                name=mode+':'+variant+':'+strategy
                run_saved(name,selected_carrier(name))
for order in ('baseline','reversed_paths'):
    for strategy in strategies:
        parent='construct:'+order+':'+strategy
        name='bids:'+order+':'+strategy;q=copy.deepcopy(inputs['requests'][name]['request'])
        assert q['inputs']['optimized_book']=={}
        q['inputs']['optimized_book']=copy.deepcopy(returned[parent]['optimized_book'])
        q['inputs']['signal_panel']=copy.deepcopy(composed_requests[parent]['inputs']['signal_panel'])
        q['inputs']['scenario_exposures']=copy.deepcopy(composed_requests[parent]['inputs']['scenario_exposures'])
        run_saved(name,q)
        bid_name=name;name='awards:'+order+':'+strategy;q=copy.deepcopy(inputs['requests'][name]['request'])
        assert q['inputs']['bid_blocks']=={}
        q['inputs']['bid_blocks']=copy.deepcopy(returned[bid_name]['bid_blocks'])
        assert q['inputs']['bid_blocks']['requested_positions']==returned[parent]['optimized_book']['positions']
        joint=q['inputs']['joint_award_value_scenarios'];original=composed_requests[parent]['inputs']['candidate_selection']['original_path_indices']
        joint['signal_panel']=copy.deepcopy(composed_requests[parent]['inputs']['signal_panel'])
        joint['scenario_exposures']=copy.deepcopy(composed_requests[parent]['inputs']['scenario_exposures'])
        joint['auctions']['price']=a['auction_prices'][:,original].tolist()
        run_saved(name,q)
for label in ('baseline','cap20','risk0.10','risk0.50','outage25%','outage50%'):
    name='stress:'+label;q=copy.deepcopy(inputs['requests'][name]['request'])
    parent='construct:baseline:topology_robust_cvar'
    assert q['inputs']['portfolio_constraints']['base_positions']==[]
    q['inputs']['portfolio_constraints']['base_positions']=copy.deepcopy(returned[parent]['optimized_book']['positions'])
    run_saved(name,q)
def legacy_case(order):
    screen=returned['screen:'+('reversed_paths' if order=='reversed_paths' else 'baseline')]['candidate_selection']['screen']
    return dict(candidate_path_ids=screen['candidate_path_ids'],structural_path_ids=screen['structural_path_ids'],span_max_column_residual=screen['span_max_column_residual'],strategies={s:returned['reference:'+order+':'+s]['optimized_book']['stats'] for s in strategies})
def check_reference(actual):
    assert actual['candidate_path_ids']==reference['candidate_path_ids'] and actual['structural_path_ids']==reference['structural_path_ids']
    differences={}
    for name,expected in reference['strategies'].items():
        observed=actual['strategies'][name]
        assert expected['selected_path_ids']==observed['selected_path_ids']
        delta=max(abs(expected['positions_mw'][p]-observed['positions_mw'][p]) for p in expected['selected_path_ids'])
        assert delta<1e-5
        for key in ('fill_aware_predicted_mean','fill_aware_heldout_mean','fill_aware_heldout_cvar95_loss'):
            assert np.isclose(expected[key],observed[key],rtol=1e-7,atol=1e-5)
        differences[name]=dict(max_position_difference_mw=delta)
    return differences
legacy=legacy_case('baseline');parity=check_reference(legacy)
robustness={order:check_reference(legacy_case(order)) for order in ('reversed_paths','reversed_candidates')}
def assembled_books(order):
    return {s:dict(positions=np.asarray(returned['construct:'+order+':'+s]['optimized_book']['positions']),stats=returned['construct:'+order+':'+s]['optimized_book']['stats'],curves=returned['bids:'+order+':'+s]['bid_blocks']['curves'],awards=returned['awards:'+order+':'+s]['award_cash_distribution']['awards']) for s in strategies}
books=assembled_books('baseline');reordered=assembled_books('reversed_paths')
ledgers={s:copy.deepcopy(returned['awards:baseline:'+s]['award_cash_distribution']['ledger']) for s in strategies}
for ledger in ledgers.values():
    for key in ('awarded_vectors','net'):ledger[key]=np.asarray(ledger[key])
stresses={}
for label in ('baseline','cap20','risk0.10','risk0.50','outage25%','outage50%'):
    row=copy.deepcopy(returned['stress:'+label]['optimized_book'])
    for key in ('risk_definition','support_scope','global_sparse_optimality','scientific_acceptance'):row.pop(key)
    row['positions']=np.asarray(row['positions']);stresses[label]=row
assert len(legacy['structural_path_ids'])==25 and len(legacy['candidate_path_ids'])==72
expected={'ALL_N3_N10':5.,'ALL_N7_N10':25.,'ALL_N8_N10':25.}
actual=legacy['strategies']['column_addition_mean_variance']['positions_mw']
assert set(actual)==set(expected)
np.testing.assert_allclose(list(actual.values()),list(expected.values()),atol=1e-7)
print({'universe':len(a['path_ids']),'rank':25,'candidates':72,'reference_parity':parity})
print('Released column book:',actual)
print('Archive is controlled; the recorded historical market-cache hash is not a claim about a different local cache.')

## Reoptimized trials, not a global sparse optimum
The column method evaluates a newly added column after reoptimizing the book.
A full budget need not prohibit a useful replacement. Each restricted continuous
mean–variance solve has a gradient-LP upper bound on its objective gap. This
certifies the restricted solve, not the global combinatorial support search.
The older binding-budget failure is displayed from the frozen ablation JSON;
no old module is imported and no old market simulation is executed.

In [ ]:
column=legacy['strategies']['column_addition_mean_variance']
for row in column['column_addition_history']:
    d=row['restricted_solve']
    assert d['solver_success'] and d['max_constraint_violation']<1e-7
    assert d['objective_gap_upper_bound']<=d['objective_gap_tolerance']
    print({k:row[k] for k in ('step','evaluated_columns','added_path_id','best_reoptimized_improvement')})
print('Final restricted certificate:',column['restricted_solve'])
ablation=inputs['original_ablation_display_only']
for row in ablation['rows']:
    if row['input_variant']=='baseline':
        print({k:row[k] for k in ('screen','column_rule','objective','positions_mw')})

## Ordering robustness and a single corrected risk definition
Reversing paths changes array coordinates but must not change economic identity.
We replay both orders. Legacy quantile-tail means remain confined to the
explicitly labeled reference replay; current scoring and bids use fractional
empirical expected shortfall (loss positive, 95%). Negative ES is allowed.
Mean–variance selection remains a different objective from mean–ES selection.

In [ ]:
for name,book in books.items():
    positions=book['stats']['positions_mw'];other=reordered[name]['stats']['positions_mw']
    assert set(positions)==set(other)
    np.testing.assert_allclose([positions[k] for k in sorted(positions)],[other[k] for k in sorted(other)],atol=1e-7)
    print(name,{'requested_MW':book['stats']['total_requested_mw'],
        'sink_concentration':book['stats']['maximum_sink_concentration_share'],
        'legacy_heldout_ES':legacy['strategies'][name]['fill_aware_heldout_cvar95_loss'],
        'corrected_heldout_ES':book['stats']['fill_aware_heldout_cvar95_loss']})
print('Risk version:',METHOD_VERSION,'; order checks:',list(robustness))

## From requested MW to awarded cash flows
The staged fixture screens and sizes a book using approximate standalone fills,
constructs decreasing marginal-value bid blocks, then clears them jointly
against each of 24 frozen price/capacity scenarios. This is not joint optimization
of selection z, quantities q, and bid prices b; nor is it a historical ISO auction.
The sizing objective and bid utility even use different declared risk weights
(0.22 versus 0.35 for the ES strategy), although they share the same ES definition.

The ledger checks awarded MW ≤ requested MW, directional residual capacities,
and gross − acquisition = net for each auction scenario. Conditional average ES
is not the ES of the pooled auction/product distribution. Report both explicitly.
Fees, funding adjustments, collateral, existing inventory, and exits are absent.

In [ ]:
for name,ledger in ledgers.items():
    print(name,{'mean_awarded_MW':books[name]['awards']['expected_awarded_mw'],
        'mean_gross':np.mean([r['heldout_mean_gross'] for r in ledger['rows']]),
        'mean_acquisition':np.mean([r['acquisition_cost'] for r in ledger['rows']]),
        'mean_net':np.mean([r['heldout_mean_net'] for r in ledger['rows']]),
        'ES_of_mixture':ledger['mixture_es95_loss'],
        'mean_conditional_ES':ledger['mean_conditional_es95_loss']})
plt=setup_plot();names=list(books);short=['Independent','Column addition','Robust ES']
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(3)
axes[0].bar(x-.18,[books[n]['stats']['total_requested_mw'] for n in names],.36,label='Requested',color='#88C0D0')
axes[0].bar(x+.18,[books[n]['awards']['expected_awarded_mw'] for n in names],.36,label='Mean awarded',color='#A3BE8C')
axes[0].set_xticks(x,short);axes[0].set(ylabel='MW',title='Existing 24 auction scenarios');axes[0].legend()
axes[1].bar(x,[books[n]['stats']['maximum_sink_concentration_share'] for n in names],color='#B48EAD')
axes[1].set_xticks(x,short);axes[1].set(ylabel='Share of requested MW',ylim=(0,1.08),title='Sink concentration is a separate risk')
fig.tight_layout();plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
for name,color in zip(names,['#88C0D0','#EBCB8B','#A3BE8C']):
    rows=ledgers[name]['rows']
    ax.plot(range(24),[r['heldout_mean_net'] for r in rows],marker='.',label=short[names.index(name)],color=color)
ax.set(xlabel='Frozen auction scenario index',ylabel='Mean held-out net cash / product',title='Actual staged awards: outcomes vary by auction scenario')
ax.legend();fig.tight_layout();plt.show()

## One-at-a-time policy sensitivity on fixed support
Hold the robust book's selected three-path support and 72-path screen fixed.
Re-solve its continuous ES sizing LP with the original capital, exposure,
cluster and path caps. Tighten the source/sink MW cap, change the risk coefficient,
or reweight the existing two outage auction scenarios. No price, capacity, or
payoff row is generated or perturbed. The default case must reproduce the book.
These are conditional policy sensitivities, not a globally efficient frontier.

The original reduced-form sizing surface uses 48 evenly spaced retained product
rows per auction. Its Cartesian pairing is the existing approximation, not
evidence that auction state and delivery returns are independent. Increasing
outage weight here changes auction-state weights only, not delivery topology.
Held-out evaluation is descriptive; it does not select the stress parameters.

In [ ]:
settings=[('baseline',{}),('cap20',{'sink_cap':20.}),('risk0.10',{'risk_weight':.10}),
          ('risk0.50',{'risk_weight':.50}),('outage25%',{'outage_weight':.25}),('outage50%',{'outage_weight':.50})]
base=books['topology_robust_cvar']['positions']
np.testing.assert_allclose(stresses['baseline']['positions'],base,atol=1e-7)
for name,r in stresses.items():
    assert abs(r['train_es95_loss']-r['hinge_es95_loss'])<1e-5
    print(name,{k:round(float(r[k]),6) for k in ('train_mean','train_es95_loss','heldout_mean','heldout_es95_loss','max_violation','primal_dual_gap')})
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(range(len(stresses)),[r['positions'].sum() for r in stresses.values()],color='#88C0D0')
axes[0].set_xticks(range(len(stresses)),list(stresses),rotation=30,ha='right')
axes[0].set(ylabel='Requested MW',title='Fixed-support sizing response')
for name,r in stresses.items():
    axes[1].scatter(r['heldout_es95_loss'],r['heldout_mean'],color='#A3BE8C')
    if name not in ('risk0.10','risk0.50'): axes[1].annotate(name,(r['heldout_es95_loss'],r['heldout_mean']),xytext=(4,5),textcoords='offset points')
axes[1].set(xlabel='Held-out ES95 loss',ylabel='Held-out mean net',title='Reduced-form outcomes · risk variants overlap')
axes[1].margins(.25);fig.tight_layout();plt.show()

## Exercise and limits
Why can the tighter concentration cap reduce held-out tail loss and mean return
together? Inspect the MW vector before interpreting the risk change as better
diversification. Why do the two risk-weight changes leave this particular book
unchanged? Binding caps and a piecewise-linear objective can keep an optimum at
the same vertex over a range of coefficients.

The original column method concentrates at one sink; better training utility is
not universal out-of-sample superiority. No new monthly observations are implied
by overlapping product windows, and no global sparse-optimality claim is made.

In [ ]:
print('Answer scaffold: compare total size',base.sum(),'→',stresses['cap20']['positions'].sum())
def serial(value):
    if isinstance(value,np.ndarray): return value.tolist()
    if isinstance(value,np.generic): return value.item()
    raise TypeError(type(value).__name__)

In [ ]:
for control in inputs['negative_controls']:reject(control['method'],control['request'])
def plain_display(value):
    if isinstance(value,np.ndarray):return value.tolist()
    if isinstance(value,np.generic):return value.item()
    if isinstance(value,dict):return {k:plain_display(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [plain_display(v) for v in value]
    return value
checked=plain_display(dict(parity=parity,robustness=robustness,books=books,ledgers=ledgers,stresses=stresses))
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(complete_retained_bidbook=True,source_risk_conventions_preserved=True,actual_composed_parent_results=True,new_synthetic_data=False,scientific_acceptance=False))
print(summary)